# Évaluation du clustering — Analyse Silhouette

Le score silhouette mesure pour chaque point :
- **a** : distance moyenne aux points de son propre cluster
- **b** : distance moyenne aux points du cluster le plus proche
- **s = (b - a) / max(a, b)** → entre -1 (mal classé) et +1 (bien classé)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

plt.rcParams['figure.figsize'] = (12, 8)

In [ ]:
# Chargement et préparation (même pipeline que le clustering)
housing = pd.read_csv('./data/housing_data_knn.csv')
features = ['latitude', 'longitude', 'median_income', 'median_house_value']
X = housing[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Shape : {X_scaled.shape}")

## 1. Score silhouette global pour différentes valeurs de k

In [ ]:
k_range = range(2, 11)
scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    scores.append(score)
    print(f"k={k:2d}  silhouette={score:.4f}")

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(list(k_range), scores, 'bo-', linewidth=2, markersize=8)
plt.axvline(x=6, color='r', linestyle='--', label='k = 6 (choisi)')
plt.xlabel('Nombre de clusters (k)')
plt.ylabel('Score silhouette moyen')
plt.title('Score silhouette en fonction de k')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Silhouette plot détaillé pour k=6

In [ ]:
k = 6
kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
labels = kmeans.fit_predict(X_scaled)

silhouette_avg = silhouette_score(X_scaled, labels)
sample_silhouette_values = silhouette_samples(X_scaled, labels)

print(f"Score silhouette moyen (k={k}) : {silhouette_avg:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

y_lower = 10
colors = cm.tab10(np.linspace(0, 1, k))

for i in range(k):
    cluster_values = sample_silhouette_values[labels == i]
    cluster_values.sort()
    
    size = cluster_values.shape[0]
    y_upper = y_lower + size
    
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_values,
                     facecolor=colors[i], edgecolor=colors[i], alpha=0.7)
    ax.text(-0.05, y_lower + 0.5 * size, str(i), fontsize=12, fontweight='bold')
    y_lower = y_upper + 10

ax.axvline(x=silhouette_avg, color='red', linestyle='--',
           label=f'Moyenne = {silhouette_avg:.3f}')
ax.set_xlabel('Score silhouette')
ax.set_ylabel('Cluster')
ax.set_title('Silhouette plot par cluster (k=6)')
ax.set_yticks([])
ax.legend(loc='best')
plt.tight_layout()
plt.show()

## 3. Statistiques par cluster

In [ ]:
housing['cluster'] = labels
housing['silhouette'] = sample_silhouette_values

cluster_sil = housing.groupby('cluster').agg(
    effectif=('cluster', 'size'),
    silhouette_moyenne=('silhouette', 'mean'),
    silhouette_min=('silhouette', 'min'),
    pct_negatif=('silhouette', lambda x: (x < 0).mean() * 100)
).round(4)

cluster_sil.columns = ['Effectif', 'Silhouette moy.', 'Silhouette min.', '% négatif']
cluster_sil

## 4. Interprétation

- Un score silhouette moyen **> 0.5** indique une bonne structure de clustering
- Un score **entre 0.25 et 0.5** indique une structure faible mais existante
- Un score **< 0.25** suggère que les clusters se chevauchent
- Les points avec un score **négatif** sont probablement mal assignés

Les clusters avec un pourcentage élevé de scores négatifs ou une silhouette moyenne basse sont les plus fragiles et pourraient bénéficier d'un découpage différent.